# 08 — Frozen out-of-sample evaluation

Load the selected FNO checkpoint and compare it on identical untouched test targets against train-only seasonal climatology, persistence, linear trend, validation-selected EOF+ridge, and FNO.

The primary paired contrast is

\[
d_t = RMSE_{FNO,t}-RMSE_{pers,t}.
\]

Negative values favor FNO. Because daily forecast errors are serially dependent, uncertainty is estimated with a **moving-block bootstrap**, not an IID bootstrap.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from oisst_fno.baselines import EOFRidgeForecaster, seasonal_climatology
from oisst_fno.data import (
    ForecastSpec,
    SSTWindowDataset,
    Standardizer,
    forecast_target_times,
    open_oisst,
    temporal_split,
)
from oisst_fno.metrics import (
    anomaly_correlation,
    daily_rmse,
    mae,
    moving_block_bootstrap_mean_ci,
    rmse,
    skill_score,
)
from oisst_fno.model import FNO2d

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG_PATH = ROOT / "artifacts" / "metrics" / "fno_experiment.json"
BASELINE_PATH = ROOT / "artifacts" / "metrics" / "baseline_selection.json"
CHECKPOINT = ROOT / "artifacts" / "models" / "fno_best.pt"

for required in (CONFIG_PATH, BASELINE_PATH, CHECKPOINT):
    if not required.exists():
        raise FileNotFoundError(f"Missing required artifact: {required}")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
baseline_config = json.loads(BASELINE_PATH.read_text(encoding="utf-8"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPEC = ForecastSpec(config["lookback_days"], config["horizon_days"])
scaler = Standardizer(**config["scaler"])

In [ ]:
sst = open_oisst(Path(config["data_path"]))["sst"]
train_da, _, test_da = temporal_split(sst, config["train_end"], config["validation_end"])
if test_da.sizes.get("time", 0) < SPEC.lookback_days + SPEC.horizon_days:
    raise ValueError("The downloaded file does not contain a sufficiently long test period.")

target_times = forecast_target_times(test_da.time.values, SPEC)
test_ds = SSTWindowDataset(scaler.transform(test_da.values), SPEC)


def collate_with_mask(batch):
    xs, ys, masks = zip(*batch)
    x = torch.stack(xs)
    y = torch.stack(ys)
    mask = torch.stack(masks)
    return torch.cat((x, mask), dim=1), y, mask


loader = DataLoader(test_ds, batch_size=16, shuffle=False, collate_fn=collate_with_mask)
model = FNO2d(**config["model"]).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE, weights_only=True))
model.eval()

In [ ]:
fno_standardized = []
truth_standardized = []
histories_standardized = []
masks = []

with torch.no_grad():
    for x, y, mask in loader:
        prediction = model(x.to(DEVICE)).cpu().numpy()[:, 0]
        fno_standardized.append(prediction)
        truth_standardized.append(y.numpy()[:, 0])
        histories_standardized.append(x.numpy()[:, : SPEC.lookback_days])
        masks.append(mask.numpy()[:, 0].astype(bool))

fno_z = np.concatenate(fno_standardized)
truth_z = np.concatenate(truth_standardized)
history_z = np.concatenate(histories_standardized)
mask = np.concatenate(masks)

fno_c = scaler.inverse_transform(fno_z)
truth_c = scaler.inverse_transform(truth_z)
persistence_c = scaler.inverse_transform(history_z[:, -1])
slope_z = (history_z[:, -1] - history_z[:, 0]) / max(SPEC.lookback_days - 1, 1)
trend_c = scaler.inverse_transform(history_z[:, -1] + SPEC.horizon_days * slope_z)

seasonal_c = seasonal_climatology(
    train_da.values,
    train_da.time.values,
    target_times,
    half_window_days=int(baseline_config["seasonal_climatology_half_window_days"]),
)

eof_cfg = baseline_config["eof_ridge"]
eof_model = EOFRidgeForecaster(
    n_components=int(eof_cfg["n_components"]), alpha=float(eof_cfg["alpha"])
).fit(train_da.values, SPEC)
eof_c = eof_model.predict_series(test_da.values)

In [ ]:
persistence_rmse = rmse(persistence_c, truth_c, mask)
predictions = {
    "seasonal_climatology": seasonal_c,
    "persistence": persistence_c,
    "linear_trend": trend_c,
    "eof_ridge": eof_c,
    "fno": fno_c,
}

rows = []
for name, prediction in predictions.items():
    model_rmse = rmse(prediction, truth_c, mask)
    rows.append(
        {
            "model": name,
            "rmse_c": model_rmse,
            "mae_c": mae(prediction, truth_c, mask),
            "anomaly_correlation": anomaly_correlation(prediction, truth_c, seasonal_c, mask),
            "persistence_skill": skill_score(model_rmse, persistence_rmse),
        }
    )

results = pd.DataFrame(rows).sort_values("rmse_c")
results

In [ ]:
fno_daily = daily_rmse(fno_c, truth_c, mask)
persistence_daily = daily_rmse(persistence_c, truth_c, mask)
paired_delta = fno_daily - persistence_daily

mean_delta, ci_lower, ci_upper = moving_block_bootstrap_mean_ci(
    paired_delta,
    block_length=min(28, len(paired_delta)),
    n_bootstrap=4000,
    confidence=0.95,
    seed=42,
)

paired_summary = {
    "contrast": "daily_rmse_fno_minus_persistence_c",
    "mean_difference_c": mean_delta,
    "block_length_days": min(28, len(paired_delta)),
    "bootstrap_samples": 4000,
    "confidence": 0.95,
    "ci_lower_c": ci_lower,
    "ci_upper_c": ci_upper,
    "robust_fno_improvement_at_95pct": bool(ci_upper < 0.0),
}
paired_summary

In [ ]:
metrics_dir = ROOT / "artifacts" / "metrics"
predictions_dir = ROOT / "artifacts" / "predictions"
metrics_dir.mkdir(parents=True, exist_ok=True)
predictions_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(metrics_dir / "test_metrics.csv", index=False)

evaluation = {
    "models": rows,
    "paired_fno_vs_persistence": paired_summary,
    "validation_defined_hard_persistence_threshold_c": baseline_config[
        "validation_persistence_daily_rmse_q75_c"
    ],
}
(metrics_dir / "test_metrics.json").write_text(json.dumps(evaluation, indent=2), encoding="utf-8")

np.savez_compressed(
    predictions_dir / "test_predictions.npz",
    target_dates=target_times.astype("datetime64[D]").astype(str),
    lat=test_da.lat.values,
    lon=test_da.lon.values,
    mask=mask.astype(np.uint8),
    truth=truth_c.astype(np.float32),
    fno=fno_c.astype(np.float32),
    persistence=persistence_c.astype(np.float32),
    linear_trend=trend_c.astype(np.float32),
    seasonal_climatology=seasonal_c.astype(np.float32),
    eof_ridge=eof_c.astype(np.float32),
    persistence_daily_rmse=persistence_daily.astype(np.float32),
)
print(metrics_dir / "test_metrics.json")
print(predictions_dir / "test_predictions.npz")

In [ ]:
def spatial_rmse(prediction, target, valid_mask):
    squared = np.where(valid_mask, (prediction - target) ** 2, np.nan)
    return np.sqrt(np.nanmean(squared, axis=0))

fno_map = spatial_rmse(fno_c, truth_c, mask)
persistence_map = spatial_rmse(persistence_c, truth_c, mask)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(fno_map - persistence_map, origin="lower", aspect="auto")
ax.set_title("FNO RMSE minus persistence RMSE by grid cell")
fig.colorbar(im, ax=ax, label="Δ RMSE (°C); negative favors FNO")
plt.show()

## Interpretation rule

A positive aggregate persistence-skill score is **not sufficient**.

The FNO case becomes substantially stronger only if:

1. the paired mean daily RMSE difference is negative;
2. its block-bootstrap interval is consistent with an improvement;
3. FNO also compares favorably with EOF+ridge and learned baselines;
4. notebook 09 identifies where the error reduction occurs in frequency/season rather than hiding it in one scalar.

If the interval crosses zero, describe the aggregate advantage as uncertain on this holdout period.